## Modbus IDS (unsupervised) — CSV-trained

This notebook trains an **unsupervised** anomaly detector from the exported Modbus CSV (`Initial Dataset 4-14/...csv`).

**Dependencies:** `pandas`, `numpy`, `scikit-learn`, `matplotlib`


Install the Python packages used for data handling, modeling, and plots.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib

Import libraries and set **paths** to your CSV and JSONL files, **window length** (`WINDOW_SEC`), and toggles for which feature groups to use later.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('../Initial Dataset 4-14')
CSV_PATH = DATA_DIR / 'modbus_dataset_2026-04-14 13_17_14.116610.csv'
JSONL_PATHS = [
    DATA_DIR / 'ids_events_orch1.jsonl',
    DATA_DIR / 'ids_events_orch2.jsonl',
    DATA_DIR / 'ids_events_orch3.jsonl',
]

# Windowing
WINDOW_SEC = 10.0

# Feature choice toggles
DROP_SRC_IDENTITY = True  # drop features that strongly encode who talked (recommended)
INCLUDE_DST_CONTEXT = True  # keep per-window dst diversity (often useful in OT)


Load the Modbus export into a **pandas DataFrame** and preview the first rows.

In [ ]:
df = pd.read_csv(CSV_PATH)
df.head()

Parse **`Time`** to datetimes, coerce numeric columns (`Func`, `Unit_ID`, `Trans_ID`), and add **`ts`** as epoch seconds for windowing and inter-arrival math.

In [ ]:
# Parse time and normalize columns
df['Time'] = pd.to_datetime(df['Time'], errors='coerce')
df = df.dropna(subset=['Time'])

# Normalize common fields
df['Func'] = pd.to_numeric(df['Func'], errors='coerce')
df['Unit_ID'] = pd.to_numeric(df['Unit_ID'], errors='coerce')
df['Trans_ID'] = pd.to_numeric(df['Trans_ID'], errors='coerce')

df['Type'] = df['Type'].astype(str)
df['Dst IP'] = df['Dst IP'].astype(str)

# Epoch seconds (UTC-naive; use JSONL overlay only as a rough check unless you align timezones)
df['ts'] = df['Time'].astype('int64') / 1e9

df[['Time', 'Dst IP', 'Trans_ID', 'Unit_ID', 'Func', 'Type']].head()

Assign each row to a **time window index** `w` using `WINDOW_SEC`, anchored at the first timestamp `t0`.

In [ ]:
# Window assignment
t0 = float(df['ts'].min())
df['w'] = ((df['ts'] - t0) // WINDOW_SEC).astype(int)
df[['Time', 'w', 'Dst IP', 'Func', 'Type']].head()

Aggregate each window into **behavioral features**: message counts, request/response mix, FC entropy and per-FC counts, transaction-ID stats, unit diversity, and inter-arrival statistics. Output is one row per window in **`feat`**.

In [ ]:
def shannon_entropy(counts: np.ndarray) -> float:
    counts = counts.astype(float)
    s = counts.sum()
    if s <= 0:
        return 0.0
    p = counts / s
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())


def build_window_features(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for w, g in df.groupby('w'):
        # Basic counts
        n_msgs = len(g)
        n_req = int((g['Type'] == 'Request').sum())
        n_resp = int((g['Type'] == 'Response').sum())

        # Function codes
        func_counts = g['Func'].value_counts(dropna=True)
        n_unique_func = int(func_counts.shape[0])
        func_entropy = shannon_entropy(func_counts.values)

        # Direction-ish: in your CSV, 'Dst IP' is the destination for both requests and responses.
        # We treat dst diversity as context, not identity.
        n_unique_dst = int(g['Dst IP'].nunique())

        # Transaction IDs: scan/recon tends to create odd patterns
        trans_counts = g['Trans_ID'].value_counts(dropna=True)
        trans_reuse_max = int(trans_counts.max()) if len(trans_counts) else 0
        trans_entropy = shannon_entropy(trans_counts.values) if len(trans_counts) else 0.0

        # Unit IDs
        unit_counts = g['Unit_ID'].value_counts(dropna=True)
        n_unique_unit = int(unit_counts.shape[0])

        # Inter-arrival time stats (message-level)
        ts_sorted = np.sort(g['ts'].values)
        if len(ts_sorted) >= 2:
            iats = np.diff(ts_sorted)
            iat_mean = float(iats.mean())
            iat_std = float(iats.std())
            iat_p95 = float(np.percentile(iats, 95))
        else:
            iat_mean = 0.0
            iat_std = 0.0
            iat_p95 = 0.0

        row = {
            'w': int(w),
            't_start': float(w * WINDOW_SEC + t0),
            't_end': float((w + 1) * WINDOW_SEC + t0),
            'n_msgs': n_msgs,
            'n_req': n_req,
            'n_resp': n_resp,
            'req_ratio': float(n_req / max(1, n_msgs)),
            'n_unique_func': n_unique_func,
            'func_entropy': func_entropy,
            'n_unique_unit': n_unique_unit,
            'n_unique_dst': n_unique_dst,
            'trans_reuse_max': trans_reuse_max,
            'trans_entropy': trans_entropy,
            'iat_mean': iat_mean,
            'iat_std': iat_std,
            'iat_p95': iat_p95,
        }

        # Add a small FC histogram for common Modbus codes
        for fc in [1, 2, 3, 4, 5, 6, 15, 16]:
            row[f'fc_{fc}_cnt'] = int((g['Func'] == fc).sum())
        rows.append(row)

    out = pd.DataFrame(rows).sort_values('w').reset_index(drop=True)
    return out


feat = build_window_features(df)
feat.head()

Select **numeric feature columns** for training (drop window id and time bounds), apply toggles (e.g. drop destination-diversity if you disable `INCLUDE_DST_CONTEXT`), and build matrix **`X`** with NaN/inf cleaned.

In [ ]:
# Prepare X
feature_cols = [c for c in feat.columns if c not in ('w', 't_start', 't_end')]

if DROP_SRC_IDENTITY:
    # Nothing in this feature set is raw src ip, but keep the switch for future extensions.
    pass

if not INCLUDE_DST_CONTEXT:
    feature_cols = [c for c in feature_cols if c not in ('n_unique_dst',)]

X = feat[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
X.shape, len(feature_cols)

**Standardize** features, fit an **Isolation Forest** (unsupervised), and attach **`if_score`** (higher = more normal) and **`if_pred`** (+1 inlier / −1 outlier) to each window.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

scaler = StandardScaler()
Xz = scaler.fit_transform(X)

# contamination='auto' picks a threshold; you can set a fixed rate later
model = IsolationForest(
    n_estimators=300,
    contamination='auto',
    random_state=42,
    n_jobs=-1,
)
model.fit(Xz)

# sklearn convention: higher decision_function = more normal
feat['if_score'] = model.decision_function(Xz)
feat['if_pred'] = model.predict(Xz)  # 1=inlier, -1=outlier

feat[['t_start', 'if_score', 'if_pred']].head()

### Reading “anomalies” from Isolation Forest

- **`if_pred`**: `+1` = normal (inlier), **`-1` = anomaly (outlier)** for that time window.
- **`if_score`**: higher = *more* like the bulk of the data; **lower / more negative = more suspicious**.
- The line plot alone does not spell out “anomaly”; you need to look at **`if_pred`** or scores in the tails.
- With **`contamination='auto'`**, sklearn picks an internal expected outlier rate; if almost everything looks “normal”, try a fixed **`contamination`** (e.g. `0.05`) or inspect the **lowest** scores below.

In [ ]:
n_total = len(feat)
n_anom = int((feat["if_pred"] == -1).sum())
print(f"Time windows: {n_total}")
print(f"Flagged as anomaly (if_pred == -1): {n_anom} ({100.0 * n_anom / max(1, n_total):.2f}%)")
print(f"Normal (if_pred == +1): {n_total - n_anom}")

show_cols = [c for c in ["w", "t_start", "if_score", "if_pred", "n_msgs", "n_unique_func", "func_entropy", "n_unique_dst"] if c in feat.columns]
print("\nMost anomalous windows (lowest if_score):")
print(feat.sort_values("if_score", ascending=True).head(15)[show_cols].to_string())

if n_anom > 0:
    print("\nRows with if_pred == -1:")
    print(feat.loc[feat["if_pred"] == -1, show_cols].sort_values("if_score").to_string())

Plot **Isolation Forest score vs time**; **red points** are windows with **`if_pred == -1`** (model’s anomaly flag).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3.5))
ax.plot(feat["t_start"], feat["if_score"], lw=0.8, color="0.65", label="score (all windows)")
anomaly_mask = feat["if_pred"] == -1
if anomaly_mask.any():
    ax.scatter(
        feat.loc[anomaly_mask, "t_start"],
        feat.loc[anomaly_mask, "if_score"],
        color="crimson",
        s=22,
        zorder=5,
        label="anomaly (if_pred=-1)",
    )
ax.set_title("Isolation Forest: score vs time (higher = more normal)")
ax.set_xlabel("epoch seconds (from CSV)")
ax.set_ylabel("if_score")
ax.legend(loc="upper right", fontsize=8)
plt.tight_layout()
plt.show()

## Optional: overlay orchestrator JSONL

Your JSONL timestamps are UTC (`ts_iso`). Your CSV `Time` may be local time. If the red lines do not align, convert one side so both use the same timezone.

Read all JSONL files and collect **`episode_start`** times as Unix epoch seconds, plus **`attack_id`** and **`source_ip`**, for optional evaluation overlays.

In [ ]:
from datetime import datetime

def load_episode_starts(jsonl_paths):
    starts = []
    for p in jsonl_paths:
        if not p.exists():
            continue
        with open(p, encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                o = json.loads(line)
                if o.get('event') != 'episode_start':
                    continue
                ts_iso = o.get('ts_iso')
                if not ts_iso:
                    continue
                # ts_iso includes offset, so timestamp() is epoch seconds
                t = datetime.fromisoformat(ts_iso.replace('Z', '+00:00')).timestamp()
                starts.append((t, o.get('attack_id', ''), o.get('source_ip', '')))
    return sorted(starts, key=lambda x: x[0])

episode_starts = load_episode_starts(JSONL_PATHS)
len(episode_starts), episode_starts[:3]

Redraw the IF score series and add **vertical lines** at each `episode_start` to sanity-check alignment between labels and anomaly scores.

In [ ]:
if episode_starts:
    fig, ax = plt.subplots(figsize=(14, 3))
    ax.plot(feat['t_start'], feat['if_score'], lw=0.9)
    for t, attack_id, src in episode_starts:
        ax.axvline(t, color='red', alpha=0.12, lw=0.8)
    ax.set_title('IF score with episode_start markers (check timezone alignment)')
    ax.set_xlabel('epoch seconds')
    ax.set_ylabel('score')
    plt.tight_layout()
    plt.show()

## Per-message (row) anomaly scores

The window model flags **time buckets**. For **individual messages** (one CSV row each), we build a small feature vector **per row** (function code, unit, transaction id, request vs response, time since previous message, text length of `Data`), then run a **separate** Isolation Forest. Output: **`pkt_if_score`** / **`pkt_if_pred`** on the sorted copy **`pkt`**.

Caveat: “anomalous row” means **unusual vs the whole capture**, not guaranteed malicious. Rare but legitimate traffic (e.g. one-off FC) can score high.

Configure **packet-level** detector: expected outlier fraction (`PACKET_CONTAMINATION`), whether to encode **`Dst IP`** as a numeric column (can encourage IP-based shortcuts), and tree count for speed.

In [ ]:
PACKET_CONTAMINATION = 0.01  # try 0.005–0.05; or None for sklearn 'auto'
PACKET_INCLUDE_DST = False  # True = include factorized Dst IP (stronger identity signal)
PACKET_N_ESTIMATORS = 150

Sort messages by time, compute **inter-arrival** `iat_sec`, build **`X_pkt`**, standardize, fit IF, attach **`pkt_if_score`** (higher = more normal) and **`pkt_if_pred`** (−1 = anomaly).

In [ ]:
pkt = df.sort_values("ts").reset_index(drop=True)
pkt["iat_sec"] = pkt["ts"].diff().fillna(0.0).clip(lower=0.0)
pkt["is_req"] = (pkt["Type"] == "Request").astype(np.int8)
pkt["data_len"] = pkt["Data"].astype(str).str.len()

pkt_feat_cols = ["Func", "Unit_ID", "Trans_ID", "is_req", "iat_sec", "data_len"]
if PACKET_INCLUDE_DST:
    pkt["dst_code"], _ = pd.factorize(pkt["Dst IP"], sort=True)
    pkt_feat_cols.append("dst_code")

X_pkt = pkt[pkt_feat_cols].replace([np.inf, -np.inf], np.nan).fillna(0.0)
scaler_pkt = StandardScaler()
X_pkt_z = scaler_pkt.fit_transform(X_pkt)

if_model_pkt = IsolationForest(
    n_estimators=PACKET_N_ESTIMATORS,
    contamination=PACKET_CONTAMINATION if PACKET_CONTAMINATION is not None else "auto",
    random_state=42,
    n_jobs=-1,
)
if_model_pkt.fit(X_pkt_z)
pkt["pkt_if_score"] = if_model_pkt.decision_function(X_pkt_z)
pkt["pkt_if_pred"] = if_model_pkt.predict(X_pkt_z)

Print **how many** rows are flagged and show the **most anomalous** messages (lowest `pkt_if_score`) with key columns for analyst review.

In [ ]:
n_pkt = len(pkt)
n_pkt_anom = int((pkt["pkt_if_pred"] == -1).sum())
print(f"Messages: {n_pkt} | Flagged anomaly (pkt_if_pred==-1): {n_pkt_anom} ({100*n_pkt_anom/max(1,n_pkt):.3f}%)")

show_pkt = ["Time", "Dst IP", "Func", "Type", "Unit_ID", "Trans_ID", "iat_sec", "data_len", "pkt_if_score", "pkt_if_pred"]
show_pkt = [c for c in show_pkt if c in pkt.columns]
top_n = 40
worst = pkt.sort_values("pkt_if_score", ascending=True).head(top_n)
print(f"\nTop {top_n} most anomalous messages (lowest pkt_if_score):")
print(worst[show_pkt].to_string())

if "Data" in pkt.columns:
    print("\nSample Data field for top 10:")
    for i, row in worst.head(10).iterrows():
        d = str(row["Data"])[:120]
        print(f"  {row['Time']} | FC={row.get('Func')} | {d}")

Optional: **histogram** of `pkt_if_score` (most messages cluster on the right; tails on the left are rarer / more anomalous).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(pkt["pkt_if_score"], bins=80, color="0.4", edgecolor="white")
ax.set_xlabel("pkt_if_score (higher = more normal)")
ax.set_ylabel("count")
ax.set_title("Per-message Isolation Forest scores")
plt.tight_layout()
plt.show()

## Evaluation vs JSONL (attack time windows)

We treat **`episode_start` → `episode_end`** pairs (per `attack_id`) as **ground-truth intervals** when attack scripts were running. A packet is **label = 1** if its timestamp falls inside any interval, else **0**.

**Prediction:** `pkt_if_pred == -1` is the model’s “anomalous message” flag.

**Why ~1% flagged?** With `PACKET_CONTAMINATION = 0.01`, Isolation Forest is asked to mark about **1%** of rows as outliers — that is a **hyperparameter**, not the true attack rate. For evaluation you compare to JSONL; to tune operating point, vary contamination or **threshold on `pkt_if_score`** and recompute metrics / ROC.

**Caveats:** (1) CSV time vs JSONL **`ts_iso`** must share the same timeline; if overlays misalign, set `EVAL_TS_OFFSET_SEC`. (2) Benign traffic still occurs during an attack window, so perfect precision/recall is unlikely.


Load **attack intervals** from each JSONL (pair start/end by `attack_id`). Optionally shift packet times for clock/timezone alignment.

In [ ]:
from collections import defaultdict, deque
from datetime import datetime

# If JSONL lines do not line up with CSV timestamps, adjust (seconds added to pkt['ts'])
EVAL_TS_OFFSET_SEC = 0.0


def _iso_to_epoch(s):
    if not s:
        return None
    return datetime.fromisoformat(str(s).replace("Z", "+00:00")).timestamp()


def load_attack_intervals_epoch(path):
    """Return list of (t0_epoch, t1_epoch, attack_id) from one JSONL file."""
    pending = defaultdict(deque)
    out = []
    if not path.exists():
        return out
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            o = json.loads(line)
            ev = o.get("event")
            ts = _iso_to_epoch(o.get("ts_iso"))
            if ts is None:
                continue
            aid = str(o.get("attack_id", ""))
            if ev == "episode_start" and aid:
                pending[aid].append(ts)
            elif ev == "episode_end" and aid:
                if not pending[aid]:
                    continue
                t0 = pending[aid].popleft()
                t1 = ts
                if t1 < t0:
                    t0, t1 = t1, t0
                out.append((t0, t1, aid))
    return out


attack_intervals = []
for jp in JSONL_PATHS:
    attack_intervals.extend(load_attack_intervals_epoch(jp))

print(f"Loaded {len(attack_intervals)} attack intervals from JSONL")
attack_intervals[:3]

Build **per-packet labels** `y_true_attack` (1 = inside any attack interval) and compare to **`y_pred_anom`** (`pkt_if_pred == -1`). Report confusion matrix, precision/recall/F1, and ROC-AUC / average precision on **`-pkt_if_score`** (higher = more anomalous).

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
    roc_auc_score,
)

ts_eval = pkt["ts"].values.astype(np.float64) + float(EVAL_TS_OFFSET_SEC)

if not attack_intervals:
    print("No attack intervals — check JSONL paths / format.")
else:
    inside = np.zeros(len(ts_eval), dtype=bool)
    for t0, t1, _aid in attack_intervals:
        inside |= (ts_eval >= t0) & (ts_eval <= t1)
    y_true = inside.astype(np.int8)
    y_pred = (pkt["pkt_if_pred"].values == -1).astype(np.int8)
    # score: higher = more anomalous
    score_anom = -pkt["pkt_if_score"].values.astype(np.float64)

    pos_rate = float(y_true.mean())
    pred_rate = float(y_pred.mean())
    print(f"Ground-truth attack-interval packets: {int(y_true.sum())} / {len(y_true)} ({100*pos_rate:.3f}%)")
    print(f"Predicted anomaly (IF):               {int(y_pred.sum())} / {len(y_pred)} ({100*pred_rate:.3f}%)")
    print()
    print("Confusion matrix [rows=true 0/1, cols=pred 0/1]:")
    print(confusion_matrix(y_true, y_pred, labels=[0, 1]))
    print()
    # positive label = 1 = attack-interval packet
    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1],
            target_names=["not_in_interval", "in_attack_interval"],
            digits=3,
            zero_division=0,
        )
    )
    # Precision/recall for the anomaly detector's positive class (pred=1) vs attack interval
    # Map: treat predicted anomaly as "alert"
    print("Rows are **ground-truth in-interval vs not**; predicted anomaly = IF flag.")
    p, r, f, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    print(f"Binary P/R/F1 (pos_label=in_attack_interval, pred=anomaly flag): P={p:.4f} R={r:.4f} F1={f:.4f}")

    if y_true.min() == y_true.max():
        print("\nROC-AUC / AP skipped: only one class in ground-truth labels.")
    else:
        try:
            roc = roc_auc_score(y_true, score_anom)
            ap = average_precision_score(y_true, score_anom)
            print(f"\nROC-AUC (score = -pkt_if_score, higher=more anomalous): {roc:.4f}")
            print(f"Average precision (PR-AUC):                        {ap:.4f}")
        except ValueError as e:
            print("ROC/AP error:", e)

### Optional: window-level metrics (same logic as `feat`)

A window is **positive** if its `[t_start, t_end]` overlaps any attack interval (not only $\geq$ contamination).

In [ ]:
def intervals_overlap(a0, a1, b0, b1):
    return (a1 >= b0) and (b1 >= a0)


try:
    _feat_ok = feat is not None and len(feat) > 0
except NameError:
    _feat_ok = False

if attack_intervals and _feat_ok:
    off = float(EVAL_TS_OFFSET_SEC)
    wt = []
    for _, row in feat.iterrows():
        a0 = float(row["t_start"]) + off
        a1 = float(row["t_end"]) + off
        hit = any(intervals_overlap(a0, a1, t0, t1) for t0, t1, _ in attack_intervals)
        wt.append(1 if hit else 0)
    yw_true = np.array(wt, dtype=np.int8)
    yw_pred = (feat["if_pred"].values == -1).astype(np.int8)
    print("Window confusion matrix [true x pred]:")
    print(confusion_matrix(yw_true, yw_pred, labels=[0, 1]))
    print(classification_report(yw_true, yw_pred, labels=[0, 1], target_names=["win_ok", "win_attack_overlap"], digits=3, zero_division=0))
else:
    print("Skip window metrics (missing feat or intervals).")